# Task 4: Other kinds of LSTM
In the first notebook we used a simple LSTM that reads the message
from the first word to the last.
Here we try two other kinds:
- Bidirectional: two LSTMs, one reads from the start to the end,
  and one reads from the end to the start.
- Stacked: two LSTM layers, one above the other.
We compare them on the validation part, not on the test data.

In [2]:
import tensorflow as tf
import pandas as pd
import numpy as np
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
train = pd.read_csv('train_prep_v2.csv')
test = pd.read_csv('test_prep_v2.csv')
print(train.shape, test.shape)

TensorFlow: 2.21.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
(4102, 9) (1026, 9)


## 1. Prepare the data in the same way
We repeat the same steps as the first LSTM notebook:
the same word numbers, the same length (40 words),
and the same validation part.
This way the comparison is fair.

In [3]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
MAX_LEN = 40
tok = Tokenizer(oov_token='<UNK>', filters='')
tok.fit_on_texts(train['clean_text'])
X_train = pad_sequences(tok.texts_to_sequences(train['clean_text']),
                        maxlen=MAX_LEN, padding='post', truncating='post')
X_test = pad_sequences(tok.texts_to_sequences(test['clean_text']),
                       maxlen=MAX_LEN, padding='post', truncating='post')
y_train = (train['label'] == 'spam').astype(int).values
y_test = (test['label'] == 'spam').astype(int).values
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)
vocab_size = len(tok.word_index) + 1
print('Words:', vocab_size)
print('Fit  :', X_fit.shape, '| spam:', y_fit.sum())
print('Valid:', X_val.shape, '| spam:', y_val.sum())

Words: 7305
Fit  : (3281, 40) | spam: 401
Valid: (821, 40) | spam: 100


## 2. One function to train any model
We will try three models.
To keep everything the same, we write one function
that trains a model and gives back its result on the validation part.

In [5]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import f1_score, precision_score, recall_score

def train_model(layers, name):
    tf.random.set_seed(42)
    np.random.seed(42)
    model = Sequential(layers)
    model.compile(loss='binary_crossentropy', optimizer='adam',
                  metrics=['accuracy'])
    stop = EarlyStopping(monitor='val_loss', patience=3,
                         restore_best_weights=True)
    hist = model.fit(X_fit, y_fit, validation_data=(X_val, y_val),
                     epochs=20, batch_size=32, callbacks=[stop], verbose=0)
    prob = model.predict(X_val, verbose=0).flatten()
    pred = (prob >= 0.5).astype(int)
    best = int(np.argmin(hist.history['val_loss'])) + 1
    print(name, '| epochs:', len(hist.history['loss']), '| best epoch:', best)
    return [name, model.count_params(), len(hist.history['loss']),
            round(100 * precision_score(y_val, pred), 1),
            round(100 * recall_score(y_val, pred), 1),
            round(100 * f1_score(y_val, pred), 1)]

## 3. Train the three models
- Simple LSTM: the same model as the first notebook.
- Bidirectional LSTM: one LSTM reads the message from the start,
  and another reads it from the end. Then we join the two.
- Stacked LSTM: two LSTM layers. The first one gives an output for every word
  (return_sequences=True), and the second one reads these outputs.

In [6]:
from tensorflow.keras.layers import Embedding, LSTM, Dropout, Dense, Input
from tensorflow.keras.layers import Bidirectional

simple = [Input(shape=(MAX_LEN,)), Embedding(vocab_size, 32),
          LSTM(32), Dropout(0.3), Dense(1, activation='sigmoid')]
bidir = [Input(shape=(MAX_LEN,)), Embedding(vocab_size, 32),
         Bidirectional(LSTM(32)), Dropout(0.3), Dense(1, activation='sigmoid')]
stacked = [Input(shape=(MAX_LEN,)), Embedding(vocab_size, 32),
           LSTM(32, return_sequences=True), LSTM(32),
           Dropout(0.3), Dense(1, activation='sigmoid')]

rows = []
rows.append(train_model(simple, 'Simple LSTM'))
rows.append(train_model(bidir, 'Bidirectional LSTM'))
rows.append(train_model(stacked, 'Stacked LSTM'))

Simple LSTM | epochs: 5 | best epoch: 2
Bidirectional LSTM | epochs: 7 | best epoch: 4
Stacked LSTM | epochs: 6 | best epoch: 3


## 4. Compare the three models
We compare them on the validation part.
We look at F1, because our data has much more ham than spam.

In [7]:
table = pd.DataFrame(rows, columns=['model', 'parameters', 'epochs',
                                    'precision', 'recall', 'f1'])
print(table.to_string(index=False))

             model  parameters  epochs  precision  recall   f1
       Simple LSTM      242113       5       88.5    92.0 90.2
Bidirectional LSTM      250465       7       96.8    90.0 93.3
      Stacked LSTM      250433       6       97.8    87.0 92.1


## 5. Save the results
We save the table, so Abdullah can use these numbers
in the report and in the charts,
without training the models again.

In [8]:
table.to_csv('lstm_variants.csv', index=False)
print('Saved')

Saved
